# Lesson 8: LLM + Tool Feedback Loop

## Objective
Learn how to create a multi-turn loop where LLM calls tools, receives results, and can chain multiple operations together.

## Problem Statement
**Previous lesson (Lesson 7):** LLM called one tool and we were done.

**Current problem:**
- User asks: "Add 5 and 3, then multiply by 2"
- We need LLM to call add(5,3) → get 8 → call multiply(8,2) → get 16
- How do we let the LLM see the tool result and decide what to do next?

**Solution:** Create a loop where:
1. LLM calls tool
2. Tool result is added to messages
3. LLM reads the result and decides next action
4. Repeat until LLM says "done"

## Theory Explanation

### The LLM + Tool Loop Pattern
```
LLM reads messages → Decides tool to call
              ↓
    Tool executes, creates ToolMessage
              ↓
    ToolMessage added to conversation
              ↓
    LLM reads NEW messages (with tool result)
              ↓
    LLM decides: call another tool or end?
```

### Why Does This Work?
LLMs have "in-context learning":
- Each time LLM reads messages, it sees ALL previous messages
- LLM can learn from tool results and adjust strategy
- No explicit "memory" needed - messages ARE the memory

### Convergence vs. Infinite Loops
**Risk:** What if LLM keeps calling tools forever?

**Solutions:**
1. **Max iterations**: Stop after N tool calls
2. **Stopping signal**: Train LLM to output "DONE" when finished
3. **State check**: Stop if tool resulted in final answer

## What is New in This Lesson?

**In Lesson 7:** Single tool call → END

**In Lesson 8:**
- After tool execution, route back to LLM (not to END)
- Add `max_iterations` to prevent infinite loops
- Track iteration count in state
- LLM chains multiple tool calls

## Which Imports / APIs Solve This Problem?

```python
# Same as Lesson 7, but with conditional_edges routing back to LLM
# graph.add_conditional_edges("tools", router)  # NEW: router from tools too
```

## Production Insight

In production:
1. **Token Cost**: Each loop iteration costs tokens. Track total cost.
2. **Latency**: Multi-turn flows take longer. Cache if possible.
3. **Convergence**: Always set max_iterations to prevent runaway costs
4. **Logging**: Log every tool call for debugging and billing
5. **Timeout**: Set wall-clock timeout in addition to iteration count

## Full Working Code

### Setup

In [ ]:
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from typing import TypedDict, List

load_dotenv()
vertexai.init(
    project=os.getenv("PROJECT_ID"),
    location=os.getenv("LOCATION")
)

llm = ChatVertexAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("✓ Setup complete")

### Step 1: Define Arithmetic Tools (Same as Lesson 7)

@tool
def add(a: int, b: int) -> int:
    """
    Add two numbers.
    Args:
        a: First number
        b: Second number
    Returns:
        Sum of a and b
    """
    result = a + b
    print(f"  → Tool: add({a}, {b}) = {result}")
    return result

@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two numbers.
    Args:
        a: First number
        b: Second number
    Returns:
        Product of a and b
    """
    result = a * b
    print(f"  → Tool: multiply({a}, {b}) = {result}")
    return result

@tool
def divide(a: int, b: int) -> int:
    """
    Divide two numbers (integer division).
    Args:
        a: Numerator
        b: Denominator
    Returns:
        Integer quotient
    """
    if b == 0:
        print(f"  → Tool: divide({a}, {b}) = ERROR")
        return "Error: Division by zero"
    result = a // b
    print(f"  → Tool: divide({a}, {b}) = {result}")
    return result

tools = [add, multiply, divide]
llm_with_tools = llm.bind_tools(tools)
print("✓ Defined tools and bound to LLM")

### Step 2: State Schema (with iteration tracking)

class ArithmeticState(TypedDict):
    """State for multi-turn tool calling"""
    question: str               # User's question
    messages: List              # Conversation with LLM
    current_result: int         # Last computed result
    iteration_count: int        # How many loops so far
    max_iterations: int         # Stop after this many

print("✓ Defined ArithmeticState with iteration tracking")

### Step 3: LLM Node

def llm_node(state: ArithmeticState) -> ArithmeticState:
    """
    Call LLM to decide which tool to call next.
    
    The LLM sees:
    1. The original user question
    2. All previous tool calls and results
    3. Decides next action
    """
    messages = list(state["messages"])
    iteration = state["iteration_count"]
    
    # First iteration: add user question
    if iteration == 0:
        messages.append(HumanMessage(content=state["question"]))
        print(f"  → Iteration {iteration + 1}: Added user question")
    else:
        print(f"  → Iteration {iteration + 1}: LLM re-analyzing with previous results")
    
    # Call LLM (it sees all messages including tool results from previous iteration)
    response = llm_with_tools.invoke(messages)
    messages.append(response)
    
    # Check if LLM decided to call tools
    num_tool_calls = len(response.tool_calls) if hasattr(response, "tool_calls") else 0
    print(f"     LLM output: {num_tool_calls} tool call(s)")
    
    return {
        "question": state["question"],
        "messages": messages,
        "current_result": state["current_result"],
        "iteration_count": state["iteration_count"],
        "max_iterations": state["max_iterations"]
    }

print("✓ Defined llm_node")

### Step 4: Tool Execution Node

from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

def execute_tools(state: ArithmeticState) -> ArithmeticState:
    """
    Execute the tool that LLM called.
    
    ToolNode:
    1. Finds tool calls in last AI message
    2. Executes each tool
    3. Creates ToolMessage with result
    4. Returns updated messages
    """
    messages = list(state["messages"])
    
    # Execute tools and get results
    result_output = tool_node.invoke({"messages": messages})
    result_messages = result_output["messages"]
    
    # Last message is ToolMessage with the result
    last_message = result_messages[-1]
    result_value = last_message.content
    print(f"     Result: {result_value}")
    
    return {
        "question": state["question"],
        "messages": result_messages,
        "current_result": result_value,
        "iteration_count": state["iteration_count"] + 1,  # Increment counter
        "max_iterations": state["max_iterations"]
    }

print("✓ Defined execute_tools")

### Step 5: Router (NEW: Can go back to LLM)

def router(state: ArithmeticState) -> str:
    """
    Route based on:
    1. Whether LLM called a tool
    2. Whether we've hit max iterations
    
    NEW: Can route tool node back to llm_node for feedback loop!
    """
    messages = state["messages"]
    last_message = messages[-1]
    iteration = state["iteration_count"]
    max_iter = state["max_iterations"]
    
    # Check if we hit max iterations
    if iteration >= max_iter:
        print(f"  → Router: Hit max iterations ({iteration}), ending")
        return "__end__"
    
    # Check if last message has tool calls (from LLM)
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print(f"  → Router: LLM has tool calls, executing tools")
        return "tools"
    # If last message is ToolMessage (from tool execution)
    elif last_message.__class__.__name__ == "ToolMessage":
        print(f"  → Router: Tool executed, asking LLM for next action")
        return "llm_node"  # NEW: Loop back to LLM
    else:
        print(f"  → Router: LLM finished, ending")
        return "__end__"

print("✓ Defined router with feedback loop capability")

### Step 6: Build Graph with Loop

from langgraph.graph import StateGraph, START, END

graph = StateGraph(ArithmeticState)

# Add nodes
graph.add_node("llm_node", llm_node)
graph.add_node("tools", execute_tools)

# Add edges with conditional routing from both nodes
graph.add_edge(START, "llm_node")

# NEW: Conditional edges from llm_node
graph.add_conditional_edges(
    "llm_node",
    lambda state: "tools" if (hasattr(state["messages"][-1], "tool_calls") and state["messages"][-1].tool_calls) else "__end__",
    {"tools": "tools", "__end__": END}
)

# NEW: Conditional edges from tools - routes back to llm_node!
graph.add_conditional_edges(
    "tools",
    router,
    {"llm_node": "llm_node", "__end__": END}
)

arithmetic_graph = graph.compile()
print("✓ Compiled graph with feedback loop")

### Step 7: Visualize

from IPython.display import Image, display

graph_image = arithmetic_graph.get_graph().draw_mermaid_png()
display(Image(graph_image))
print("✓ Graph visualization (notice the loop!)")

### Step 8: Run Tests

# Test multi-tool scenarios
test_questions = [
    "Add 5 and 3, then multiply the result by 2",
    "What is 10 times 3?",
    "Divide 20 by 4, then add 5"
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Input: {question}")
    print(f"{'='*60}")
    
    initial_state = {
        "question": question,
        "messages": [],
        "current_result": 0,
        "iteration_count": 0,
        "max_iterations": 5  # Prevent infinite loops
    }
    
    final_state = arithmetic_graph.invoke(initial_state)
    
    print(f"\nFinal Result: {final_state['current_result']}")
    print(f"Total Iterations: {final_state['iteration_count']}")
    print(f"Total Messages: {len(final_state['messages'])}")

## Step-by-Step Code Explanation

### 1. **The Feedback Loop Pattern**
```python
graph.add_conditional_edges(
    "tools",
    router,
    {"llm_node": "llm_node", "__end__": END}  # CAN ROUTE BACK TO LLM!
)
```
**Why?**
- After tool execution, router decides if LLM needs to do more work
- If yes → route back to "llm_node"
- If no → route to "__end__"
- Creates natural feedback loop

### 2. **Iteration Tracking**
```python
"iteration_count": state["iteration_count"] + 1  # Increment after tools

if iteration >= max_iter:
    return "__end__"  # Prevent infinite loops
```
**Why?**
- Each loop costs tokens
- Must have a maximum to prevent runaway costs
- iteration_count tracks progress

### 3. **LLM Sees Tool Results**
```python
messages = list(state["messages"])  # Includes ALL previous tool results
response = llm_with_tools.invoke(messages)  # LLM reads full history
messages.append(response)  # Add new LLM response
```
**Why?**
- Each iteration, LLM sees full conversation
- LLM can reference previous tool results
- Messages form the "working memory" of the graph

### 4. **Router Checks Last Message Type**
```python
if hasattr(last_message, "tool_calls"):  # Is it an AIMessage with tool calls?
    return "tools"  # Execute the tools
elif last_message.__class__.__name__ == "ToolMessage":  # Is it a ToolMessage?
    return "llm_node"  # Ask LLM what to do next
```
**Why?**
- Different message types indicate different states
- AIMessage with tool_calls = LLM wants to call tools
- ToolMessage = tools just executed, LLM needs to see results

## Summary

**What changed from Lesson 7?**
1. Lesson 7: One tool call → END
2. Lesson 8: Multiple tool calls in sequence → feedback loop

**Architecture:**
```
LLM node
   ↓
Tools execute (sees previous results)
   ↓
Router decides: More work or done?
   ↓ (if more work)
Back to LLM node (with tool results)
```

## Why This Matters in Production

1. **Reasoning Chains**: LLM can break down complex problems into steps
2. **Self-Correction**: LLM can check tool results and retry if wrong
3. **Cost Management**: max_iterations prevents budget overruns
4. **Transparency**: Each step logged for debugging
5. **Convergence**: LLM naturally "stops" when problem solved

**Next lesson:** We'll parallelize tools - calling multiple arithmetic operations simultaneously instead of sequentially.